# Module 02: DICOM-to-BIDS Conversion with HeudiConv

**Prerequisites:** Modules 00–01 | **Estimated time:** 2–3 hours

---

## 1. Introduction

### What is HeudiConv?

[HeudiConv](https://heudiconv.readthedocs.io/) (Heuristic DICOM Converter) is a
flexible tool that converts raw DICOM files from an MRI scanner into
[NIfTI](https://nifti.nimh.nih.gov/) images organised according to the
[Brain Imaging Data Structure (BIDS)](https://bids-specification.readthedocs.io/)
specification.

Under the hood HeudiConv delegates the actual DICOM→NIfTI conversion to
[dcm2niix](https://github.com/rordenlab/dcm2niix), but it adds:

* **Automatic series discovery** — scans your DICOM tree and groups series.
* **Heuristic-based naming** — a small Python file you write (the *heuristic*)
  maps each series to the correct BIDS filename.
* **BIDS sidecar generation** — JSON metadata files are created automatically.
* **Reproducibility** — the same heuristic applied to the same DICOMs always
  produces identical output.

### Pipeline overview

```
DICOM files
    │
    ▼
HeudiConv (dry run)  ──►  seqinfo table  ──►  write heuristic.py
    │
    ▼
HeudiConv (full run)
    │
    ├── dcm2niix  ──►  .nii.gz + .json sidecars
    │
    └── BIDS layout  ──►  validated dataset
```

In [ ]:
# Standard library imports used throughout this notebook
import subprocess
import json
import os
import pathlib
import shutil
import tempfile

# Confirm HeudiConv is available
result = subprocess.run(["heudiconv", "--version"], capture_output=True, text=True)
print("HeudiConv version:", result.stdout.strip() or result.stderr.strip())

result = subprocess.run(["dcm2niix", "-v"], capture_output=True, text=True)
print("dcm2niix:", result.stdout.strip()[:60] or result.stderr.strip()[:60])

---

## 2. Understanding DICOM Directory Structure

MRI scanners typically organise their output using a three-level hierarchy:

```
dicoms/
└── sub-01/                        ← subject (participant)
    └── 20240115/                  ← session date (or session label)
        ├── 001_localiser/         ← series number + description
        │   ├── 000001.dcm
        │   └── …
        ├── 002_t1_mprage/
        │   ├── 000001.dcm
        │   └── …
        ├── 003_func_rest/
        │   ├── 000001.dcm
        │   └── …
        └── 004_fmap_magnitude/
            ├── 000001.dcm
            └── …
```

The exact layout varies by scanner vendor and site, but the pattern is
predictable enough that a simple **glob template** can locate all DICOM files
for a given subject.

### Key DICOM fields HeudiConv reads

| DICOM tag | Example value | Used for |
|-----------|--------------|----------|
| SeriesDescription | `func_rest_2mm` | Heuristic matching |
| ProtocolName | `func_rest` | Heuristic matching |
| SeriesNumber | `3` | Ordering series |
| ImageType | `ORIGINAL\PRIMARY\M` | Distinguishing mag/phase |
| RepetitionTime | `2000` | TR in milliseconds |
| EchoTime | `30` | TE in milliseconds |
| PhaseEncodingDirection | `j-` | Unwarp direction |

In [ ]:
# Helper: pretty-print a directory tree (pure Python, no external deps)
def print_tree(root: str, prefix: str = "", max_depth: int = 4, _depth: int = 0) -> None:
    """Recursively print a directory tree up to max_depth levels."""
    if _depth > max_depth:
        return
    root_path = pathlib.Path(root)
    if not root_path.exists():
        print(f"{prefix}{root_path.name}  [NOT FOUND]")
        return
    print(f"{prefix}{root_path.name}/")
    children = sorted(root_path.iterdir())
    for i, child in enumerate(children):
        connector = "└── " if i == len(children) - 1 else "├── "
        extension = "    " if i == len(children) - 1 else "│   "
        if child.is_dir():
            print_tree(str(child), prefix + extension, max_depth, _depth + 1)
        else:
            print(f"{prefix}{connector}{child.name}")


# ── Example: set DICOM_DIR to your actual DICOM root ──────────────────────────
# Adapt this path to match your data location.
DICOM_DIR = pathlib.Path(os.environ.get("FMRI_TUTORIAL_DICOM_DIR", "/data/dicoms"))
BIDS_DIR = pathlib.Path(os.environ.get("FMRI_TUTORIAL_BIDS_DIR", "/data/bids_output"))
SUBJECT_ID = "sub-01"

print(f"DICOM_DIR : {DICOM_DIR}")
print(f"BIDS_DIR  : {BIDS_DIR}")
print(f"Subject   : {SUBJECT_ID}")

if DICOM_DIR.exists():
    print("\nDICOM directory tree (first 3 levels):")
    print_tree(str(DICOM_DIR), max_depth=3)
else:
    print("\n[INFO] DICOM_DIR not found — set the FMRI_TUTORIAL_DICOM_DIR environment variable.")

---

## 3. Step 1 — Dry-Run Mode: Discovering Series

Before any conversion, run HeudiConv with the built-in `convertall` heuristic
and no output directory write. This **dry run**:

* Reads DICOM headers without writing NIfTI files.
* Writes a `dicominfo.tsv` table listing every series with its key metadata.
* Lets you inspect what is present before you write your heuristic.

### The `--dicom_dir_template` pattern

HeudiConv uses a template string where `{subject}` and (optionally) `{session}`
are filled in at runtime:

```
dicoms/{subject}/*/*.dcm
# or with explicit session
dicoms/{subject}/{session}/*/*.dcm
```

In [ ]:
import csv

# Directory where HeudiConv will write its working files
HEUDICONV_SCRATCH = pathlib.Path("/tmp/heudiconv_scratch")
HEUDICONV_SCRATCH.mkdir(parents=True, exist_ok=True)

# Template: adjust the glob pattern to match YOUR DICOM layout
dicom_template = str(DICOM_DIR / "{subject}" / "*" / "*" / "*.dcm")

dry_run_cmd = [
    "heudiconv",
    "--dicom_dir_template", dicom_template,
    "--subjects", SUBJECT_ID,
    "--heuristic", "convertall",   # built-in: convert everything
    "--outdir", str(HEUDICONV_SCRATCH),
    "--bids",
    "--dry_run",
    "--overwrite",
]

print("Running command:")
print(" ".join(dry_run_cmd))
print()

if DICOM_DIR.exists():
    result = subprocess.run(dry_run_cmd, capture_output=True, text=True)
    print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:])
else:
    print("[SKIP] DICOM directory not found — command shown above for reference.")


def load_dicominfo(scratch_dir: pathlib.Path, subject: str) -> list[dict]:
    """Read the dicominfo.tsv written by HeudiConv dry run."""
    tsv_path = scratch_dir / ".heudiconv" / subject / "info" / "dicominfo.tsv"
    if not tsv_path.exists():
        return []
    with open(tsv_path) as fh:
        reader = csv.DictReader(fh, delimiter="\t")
        return list(reader)


rows = load_dicominfo(HEUDICONV_SCRATCH, SUBJECT_ID)
if rows:
    print(f"\nFound {len(rows)} series in dicominfo.tsv:")
    for row in rows:
        print(f"  series_id={row.get('series_id','?'):>4}  "
              f"series_description={row.get('series_description','?'):<40}  "
              f"dim4={row.get('dim4','?')}")
else:
    print("\n[INFO] No dicominfo.tsv found (expected when DICOM_DIR is not set).")

---

## 4. Step 2 — Understanding the Heuristic File

The **heuristic file** is a plain Python module that HeudiConv imports at
runtime. It must define a function called `infotodict` that receives the
`seqinfo` list (one `SeqInfo` named-tuple per DICOM series) and returns a
dictionary mapping BIDS filename **templates** to lists of matching series.

### Anatomy of a heuristic file

```python
# heuristic.py  ─ minimal working example

from heudiconv.utils import SeqInfo

def create_key(template, outtype=("nii.gz",), annotation_classes=None):
    """Return a (template, outtype, annotation_classes) tuple used as a dict key."""
    if template is None or not template:
        raise ValueError("Template must not be empty.")
    return template, outtype, annotation_classes


def infotodict(seqinfo):
    """Map DICOM series to BIDS output paths."""

    # ── Define output path templates ─────────────────────────────────────────
    t1w = create_key("sub-{subject}/anat/sub-{subject}_T1w")
    bold_rest = create_key("sub-{subject}/func/sub-{subject}_task-rest_bold")
    fmap_mag = create_key("sub-{subject}/fmap/sub-{subject}_magnitude{item}")
    fmap_phase = create_key("sub-{subject}/fmap/sub-{subject}_phasediff")

    # info maps each key to a list of matching series IDs
    info = {t1w: [], bold_rest: [], fmap_mag: [], fmap_phase: []}

    for s in seqinfo:
        # s is a SeqInfo named-tuple; common fields:
        #   s.series_id, s.series_description, s.protocol_name,
        #   s.dim1, s.dim2, s.dim3, s.dim4 (number of volumes),
        #   s.TR, s.TE, s.is_motion_corrected

        desc = s.series_description.lower()

        if "t1" in desc or "mprage" in desc:
            info[t1w].append(s.series_id)

        elif "rest" in desc and s.dim4 > 1:
            info[bold_rest].append(s.series_id)

        elif "fmap" in desc and "magnitude" in desc:
            info[fmap_mag].append(s.series_id)

        elif "fmap" in desc and "phase" in desc:
            info[fmap_phase].append(s.series_id)

    return info
```

### SeqInfo fields reference

| Field | Type | Description |
|-------|------|-------------|
| `total_files_till_now` | int | Running DICOM count |
| `example_dcm_file` | str | Path to one DICOM in the series |
| `series_id` | str | Unique series identifier |
| `dcm_dir_name` | str | Directory containing DICOMs |
| `series_files` | int | Number of DICOM files |
| `patient_id` | str | Scanner patient ID |
| `protocol_name` | str | Scanner protocol name |
| `is_motion_corrected` | bool | MC flag |
| `is_derived` | bool | Derived series flag |
| `patient_age` | str | Patient age |
| `patient_sex` | str | Patient sex |
| `date` | str | Acquisition date |
| `series_description` | str | Human-readable series name |
| `sequence_name` | str | Scanner sequence name |
| `image_type` | tuple | DICOM ImageType values |
| `dim1–dim4` | int | Volume dimensions |
| `TR` | float | Repetition time (ms) |
| `TE` | float | Echo time (ms) |

In [ ]:
# Write the heuristic file to disk so we can use it in the next step

HEURISTIC_FILE = HEUDICONV_SCRATCH / "heuristic.py"

heuristic_content = '''\
"""HeudiConv heuristic for the fMRI tutorial dataset.

Adapt the pattern matching in infotodict() to your scanner protocol.
"""


def create_key(template, outtype=("nii.gz",), annotation_classes=None):
    if template is None or not template:
        raise ValueError("Template must not be empty.")
    return template, outtype, annotation_classes


def infotodict(seqinfo):
    """Map DICOM series to BIDS output paths."""

    t1w = create_key("sub-{subject}/anat/sub-{subject}_T1w")
    bold_rest = create_key("sub-{subject}/func/sub-{subject}_task-rest_bold")
    bold_task = create_key(
        "sub-{subject}/func/sub-{subject}_task-nback_run-{item:02d}_bold"
    )
    fmap_mag = create_key("sub-{subject}/fmap/sub-{subject}_magnitude{item}")
    fmap_phase = create_key("sub-{subject}/fmap/sub-{subject}_phasediff")

    info = {
        t1w: [],
        bold_rest: [],
        bold_task: [],
        fmap_mag: [],
        fmap_phase: [],
    }

    for s in seqinfo:
        desc = s.series_description.lower()

        if any(kw in desc for kw in ("t1", "mprage", "t1w")):
            info[t1w].append(s.series_id)

        elif "rest" in desc and s.dim4 and int(s.dim4) > 1:
            info[bold_rest].append(s.series_id)

        elif "nback" in desc and s.dim4 and int(s.dim4) > 1:
            info[bold_task].append(s.series_id)

        elif "fmap" in desc and "magnitude" in desc:
            info[fmap_mag].append(s.series_id)

        elif "fmap" in desc and "phase" in desc:
            info[fmap_phase].append(s.series_id)

    return info
'''

HEURISTIC_FILE.write_text(heuristic_content)
print(f"Heuristic written to: {HEURISTIC_FILE}")
print()
print(heuristic_content)

---

## 5. Step 3 — Running HeudiConv Conversion

With the heuristic in place, run HeudiConv **without** `--dry_run` to produce
the full BIDS dataset.

### Important flags

| Flag | Description |
|------|-------------|
| `--subjects` | One or more subject labels (without the `sub-` prefix by default) |
| `--dicom_dir_template` | Glob template with `{subject}` placeholder |
| `--heuristic` | Path to heuristic `.py` file |
| `--outdir` | Root output directory (BIDS dataset root) |
| `--bids` | Enable BIDS output mode |
| `--overwrite` | Overwrite existing output files |
| `--minmeta` | Write only required JSON fields (smaller sidecars) |
| `--datalad` | Track output with DataLad (optional) |

> **Tip:** HeudiConv preserves original DICOMs under `sourcedata/` by default
> when you use `--bids`. Pass `--no_datalad` to skip DataLad tracking.

In [ ]:
import time

BIDS_DIR.mkdir(parents=True, exist_ok=True)

convert_cmd = [
    "heudiconv",
    "--subjects", SUBJECT_ID,
    "--dicom_dir_template", str(DICOM_DIR / "{subject}" / "*" / "*" / "*.dcm"),
    "--heuristic", str(HEURISTIC_FILE),
    "--outdir", str(BIDS_DIR),
    "--bids",
    "--overwrite",
]

print("Conversion command:")
print(" ".join(convert_cmd))
print()

if DICOM_DIR.exists():
    t0 = time.time()
    result = subprocess.run(convert_cmd, capture_output=True, text=True)
    elapsed = time.time() - t0

    print(f"Finished in {elapsed:.1f}s  |  return code: {result.returncode}")
    print()
    output = result.stdout + result.stderr
    print(output[-4000:] if len(output) > 4000 else output)
else:
    print("[SKIP] DICOM directory not found — command shown above for reference.")

---

## 6. Step 4 — Verifying the BIDS Output

After conversion, verify that:

1. All expected NIfTI and JSON files were created.
2. Dataset-level metadata files (`dataset_description.json`, `participants.tsv`)
   exist at the root.
3. File sizes are plausible (not empty, not unexpectedly large).
4. JSON sidecars contain key fields (`RepetitionTime`, `EchoTime`, etc.).

In [ ]:
def check_bids_output(bids_dir: pathlib.Path, subject: str) -> None:
    """Check that expected BIDS files exist and report basic statistics."""

    required_root = [
        "dataset_description.json",
        "participants.tsv",
    ]

    required_subject = [
        f"{subject}/anat/{subject}_T1w.nii.gz",
        f"{subject}/anat/{subject}_T1w.json",
        f"{subject}/func/{subject}_task-rest_bold.nii.gz",
        f"{subject}/func/{subject}_task-rest_bold.json",
    ]

    print(f"Checking BIDS output in: {bids_dir}")
    print("=" * 60)

    all_ok = True
    for rel in required_root:
        path = bids_dir / rel
        status = "✓" if path.exists() else "✗ MISSING"
        size = f"{path.stat().st_size:>10,} B" if path.exists() else ""
        print(f"  {status}  {rel:<40} {size}")
        if not path.exists():
            all_ok = False

    print()
    for rel in required_subject:
        path = bids_dir / rel
        status = "✓" if path.exists() else "✗ MISSING"
        size = f"{path.stat().st_size:>10,} B" if path.exists() else ""
        print(f"  {status}  {rel:<55} {size}")
        if not path.exists():
            all_ok = False

    print()
    print("Result:", "All required files present ✓" if all_ok else "Some files are missing ✗")

    # Spot-check the BOLD sidecar JSON
    bold_json = bids_dir / f"{subject}/func/{subject}_task-rest_bold.json"
    if bold_json.exists():
        with open(bold_json) as fh:
            meta = json.load(fh)
        print()
        print("BOLD sidecar metadata (selected fields):")
        for key in ("RepetitionTime", "EchoTime", "FlipAngle", "PhaseEncodingDirection",
                    "SliceTiming", "TaskName"):
            val = meta.get(key, "<not present>")
            print(f"  {key:<30} {val}")


if BIDS_DIR.exists():
    check_bids_output(BIDS_DIR, SUBJECT_ID)
else:
    print(f"[SKIP] BIDS_DIR not found: {BIDS_DIR}")
    print("Run the conversion cell first, or set FMRI_TUTORIAL_BIDS_DIR.")

In [ ]:
# Print a tree of the BIDS output (limited depth)
if BIDS_DIR.exists():
    print(f"BIDS dataset tree ({BIDS_DIR}):")
    print_tree(str(BIDS_DIR), max_depth=4)
else:
    print("[SKIP] BIDS directory not found.")

---

## Summary

In this notebook you:

1. Verified HeudiConv and dcm2niix are installed and accessible.
2. Explored a typical DICOM directory structure.
3. Used HeudiConv's **dry-run** mode to discover series without converting data.
4. Wrote a **heuristic file** that maps `SeriesDescription` patterns to BIDS
   filenames for T1w, BOLD, and field-map acquisitions.
5. Ran a full HeudiConv conversion and verified the resulting BIDS structure.

### Next steps

* Run the batch script (`scripts/run_heudiconv_batch.sh`) to convert all subjects.
* Validate the resulting BIDS dataset in **Module 03** using `bids-validator` and
  PyBIDS.

### Exercises

1. Modify the heuristic to handle a fieldmap-free acquisition (only
   reverse-phase-encoding EPI).
2. Add session-level support by including `ses-{session}` in the BIDS path
   templates.
3. Use the `dicominfo.tsv` to identify derived series (e.g. motion-corrected)
   and explicitly exclude them from the conversion by leaving their key lists
   empty.